# Model 5 — PanDerm Base linear-probe training (official PanDerm encoder)

Uses the **official PanDerm Base ViT-B/16 encoder** and its downloaded checkpoint, with a separate scikit-learn linear classifier. We DO NOT run the repository's `linear_eval.py` because that script loads test labels during training. This notebook keeps the real test locked.

**Required:** You must obtain the actual `panderm_bb_data6_checkpoint-499.pth` weight file using the [official PanDerm repository](https://github.com/SiyuanYan1/PanDerm). The notebook cannot substitute ImageNet weights. Check the project's non-commercial academic-use license before use. Kaggle T4 + Internet (for repo).

**Dataset contract:** All real images come from the user-uploaded Kaggle v2 collection, after the approved frozen dedup/group split. Never pass the raw unsplit v2 folder into training. The `EDIT_APPROVED_...` input paths must point to your saved Kaggle split outputs. Checkpoint/validation CSVs, final-test class probabilities (only when explicitly unlocked), run logs and SHA-256 artifact inventory are saved under `OUT_DIR`.


**Run evidence:** console output is streamed into `notebook_console.log` from before model loading onward; training history and checkpoint files remain in `OUT_DIR`, with SHA-256 inventory and structured `run_log.jsonl`. All raw logs stay in private Kaggle storage until a privacy-reviewed aggregate export is prepared.


## 1. Install dependencies (do not override Kaggle torch) and clone upstream code

In [ ]:
import sys,subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'timm==0.9.16','open_clip_torch','scikit-learn>=1.4,<2','pandas','pillow','tqdm','einops'])
from pathlib import Path
REPO=Path('/kaggle/working/PanDerm')
if not (REPO/'classification'/'models'/'builder.py').is_file():
 subprocess.check_call(['git','clone','--depth','1','https://github.com/SiyuanYan1/PanDerm.git',str(REPO)])
print('PanDerm repository',REPO)

## 2. Edit dataset and PanDerm weight paths

In [ ]:
# EDIT ONLY THESE PATHS FIRST (attach Kaggle datasets using Add Input)
MANIFEST_CSV = '/kaggle/input/EDIT_APPROVED_REAL_SPLIT/real_all_manifest.csv'  # needs image_path,label,split,group_id,is_synthetic
IMAGE_ROOT = '/kaggle/input/EDIT_APPROVED_REAL_IMAGES/facial_real_split_v2'       # used for relative image paths
EXPERIMENT = 'real_only'  # 'real_only', 'real_plus_synthetic', 'synthetic_only'
RUN_FINAL_TEST = False   # change ONLY after all hyperparameters and experiment selection are frozen
SEED = 34021
HEAD_EPOCHS = 2
FINETUNE_EPOCHS = 6
BATCH_SIZE = 4
NUM_WORKERS = 2
PATIENCE = 4
OUT_DIR = '/kaggle/working/panderm_base_outputs'

PAN_DERM_WEIGHTS = '/kaggle/input/YOUR-PANDERM-WEIGHTS/panderm_bb_data6_checkpoint-499.pth' # CHANGE
C_SVM = [0.01,0.1,1.,10.] # select only on real validation; train uses ridge logistic regression

In [ ]:
# Save all notebook console text, including training metrics and any raised errors.
# Keep raw logs in private Kaggle outputs; publish only aggregate checked CSVs.
import sys,subprocess
from pathlib import Path
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1',
        'https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import start_notebook_logging
PRIVATE_CONSOLE_LOG=start_notebook_logging(OUT_DIR,kind='classifier_training',
    config={'model':'PanDerm_Base','experiment':EXPERIMENT,'seed':SEED,
            'manifest_sha256':'recorded separately in final artifact inventory',
            'final_real_test_unlocked':RUN_FINAL_TEST},repo_root=DERMSYNREAL_REPO)
print('Persistent Kaggle training console:',PRIVATE_CONSOLE_LOG)


## 3. Validate frozen real manifest, resolve image paths

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, random, json, hashlib
import torch
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report, roc_auc_score

LABELS = ['acne','fungal_infection','vitiligo','hyperpigmentation','skin_cancer','normal']
LABEL2ID = {s:i for i,s in enumerate(LABELS)}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device',DEVICE)

m = pd.read_csv(MANIFEST_CSV, dtype=str, keep_default_na=False)
required = {'image_path','label','split','group_id','is_synthetic'}
assert required <= set(m.columns), f'Missing required manifest columns: {required-set(m.columns)}'
for col in required: m[col] = m[col].astype(str).str.strip()
m['label'] = m['label'].str.lower().str.replace(' ', '_', regex=False)
m['split'] = m['split'].str.lower().replace({'validation':'val','valid':'val'})
m['is_synthetic'] = m['is_synthetic'].str.lower().map({'true':True,'1':True,'yes':True,'false':False,'0':False,'no':False})
assert m['is_synthetic'].notna().all(), 'is_synthetic must be true/false or 1/0'
assert m['label'].isin(LABELS).all(), f'Unknown class labels: {set(m.label)-set(LABELS)}'
assert m['split'].isin(['train','val','test']).all(), 'Split must be train, val or test'
assert not m['image_path'].duplicated().any(), 'Duplicate image_path entries'
assert not ((m['is_synthetic']) & (m['split']!='train')).any(), 'Synthetic data must be train only'
real = m[~m['is_synthetic']].copy()
assert not real['group_id'].eq('').any(), 'Every real image requires group_id'
assert {'train','val','test'} <= set(real['split']), 'Real train, val and test required'
group_split_counts = real.groupby('group_id')['split'].nunique()
assert group_split_counts.max() == 1, 'Patient/duplicate group crosses real partitions'
for name in ['train','val']:
 assert set(LABELS) <= set(real.loc[real.split==name,'label']), f'Missing classes in real {name}'
root = Path(IMAGE_ROOT)
def to_path(v):
 p=Path(v)
 return p if p.is_absolute() else root / p
m['resolved_path'] = m['image_path'].map(lambda v:str(to_path(v)))
missing=[p for p in m.resolved_path if not Path(p).is_file()]
assert not missing, f'Missing {len(missing)} files, examples: {missing[:3]}'
if EXPERIMENT == 'real_only':
 tr = m[(m.split=='train') & (~m.is_synthetic)].copy()
elif EXPERIMENT == 'synthetic_only':
 tr=m[(m.split=='train') & (m.is_synthetic)].copy()
 assert len(tr)>0, 'E5 needs reviewed synthetic images'
elif EXPERIMENT == 'real_plus_synthetic':
 tr = m[m.split=='train'].copy()
 assert tr.is_synthetic.any(), 'No synthetic train images in manifest'
else: raise ValueError('EXPERIMENT must be real_only, real_plus_synthetic or synthetic_only')
va = m[(m.split=='val') & (~m.is_synthetic)].copy()
te = m[(m.split=='test') & (~m.is_synthetic)].copy() # evaluated only when RUN_FINAL_TEST=True
tr['y']=tr.label.map(LABEL2ID); va['y']=va.label.map(LABEL2ID); te['y']=te.label.map(LABEL2ID)
print('Train',len(tr),'Real val',len(va),'Frozen real test',len(te),'Experiment',EXPERIMENT)

## 4. Load the official PanDerm Base encoder

In [ ]:
from argparse import Namespace
import sys
sys.path.insert(0,str(REPO/'classification'))
WEIGHTS=Path(PAN_DERM_WEIGHTS)
assert WEIGHTS.is_file(), f'Official PanDerm weight not provided: {WEIGHTS}'
# Import the original authors' encoder factory. Do not replace it with an unrelated timm model.
from models import get_encoder
args=Namespace(pretrained_checkpoint=str(WEIGHTS))
encoder, eval_transform=get_encoder(args,'PanDerm_Base_LP')
encoder=encoder.to(DEVICE).eval()
for p in encoder.parameters():p.requires_grad=False
print('PanDerm Base encoder loaded; features will be extracted without gradient updates.')

## 5. Extract PanDerm features (train and real validation only)

In [ ]:
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report, confusion_matrix
import joblib
class PanDS(Dataset):
 def __init__(self,df):self.df=df.reset_index(drop=True)
 def __len__(self):return len(self.df)
 def __getitem__(self,i):
  row=self.df.iloc[i]
  with Image.open(row.resolved_path) as f:
   x=eval_transform(ImageOps.exif_transpose(f).convert('RGB'))
  return x,LABEL2ID[row.label]
@torch.inference_mode()
def features(frame):
 dl=DataLoader(PanDS(frame),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
 feats=[];labels=[]
 for x,y in tqdm(dl):
  z=encoder.forward_features(x.to(DEVICE),is_train=False)
  assert z.ndim==2, f'Unexpected PanDerm features shape {tuple(z.shape)}'
  feats.append(z.float().cpu().numpy());labels.append(y.numpy())
 return np.vstack(feats),np.concatenate(labels)
X_tr,y_tr=features(tr);X_va,y_va=features(va)
print('Train features',X_tr.shape,'Validation features',X_va.shape)

## 6. Fit linear probe and tune on real validation only

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
best=-1;chosen=None;rows=[]
for c in C_SVM:
 clf=make_pipeline(StandardScaler(),LogisticRegression(C=c,max_iter=2000,class_weight='balanced'))
 clf.fit(X_tr,y_tr)
 yp=clf.predict(X_va)
 score=f1_score(y_va,yp,labels=list(range(6)),average='macro',zero_division=0)
 rows.append({'C':c,'val_macro_f1':score})
 print('C',c,'real val macroF1',round(score,4))
 if score>best:best=score;chosen=clf;chosen_c=c
out=Path(OUT_DIR);out.mkdir(parents=True,exist_ok=True)
pd.DataFrame(rows).to_csv(out/'val_tuning.csv',index=False)
joblib.dump(chosen,out/'linear_probe.joblib')
(out/'model_details.json').write_text(json.dumps({'encoder':'PanDerm_Base_LP','checkpoint_name':WEIGHTS.name,
 'experiment':EXPERIMENT,'val_macro_f1':best,'C':chosen_c,'labels':LABELS},indent=2))
print('Selected C',chosen_c,'val macro F1',best)

## 7. Unlock final real test ONLY after all research choices are frozen

In [ ]:
if RUN_FINAL_TEST:
 X_te,y_te=features(te)
 p=chosen.predict_proba(X_te)
 pred=chosen.predict(X_te)
 print('FINAL REAL TEST macro-F1',f1_score(y_te,pred,labels=list(range(6)),average='macro',zero_division=0))
 print(classification_report(y_te,pred,labels=list(range(6)),target_names=LABELS,zero_division=0))
 np.savez_compressed(out/'FINAL_real_test_predictions.npz',y=y_te,probabilities=p)
else:
 print('Real test set remains unused.')

In [ ]:
# Research CSV export: exact existing test-order provenance, no re-inference.
# Only runs after an explicitly enabled final real-test evaluation.
if RUN_FINAL_TEST:
    import hashlib
    artifact = np.load(out / 'FINAL_real_test_predictions.npz')
    test_frame = te.reset_index(drop=True)
    labels_np = np.asarray(artifact['y'], dtype=int)
    scores_np = np.asarray(artifact['probabilities'])
    expected_np = test_frame['y'].to_numpy(dtype=int)
    assert len(test_frame) == len(labels_np) == len(scores_np), 'Test outputs and IDs have different lengths'
    assert np.array_equal(labels_np, expected_np), 'Prediction order does not match frozen real-test manifest'
    assert scores_np.ndim == 2 and scores_np.shape[1] == len(LABELS), 'Unexpected probability shape'
    assert np.all(np.isfinite(scores_np)), 'Nonfinite probabilities detected'
    predictions_table = pd.DataFrame({
        'image_id': test_frame['image_id'].astype(str).to_list() if 'image_id' in test_frame.columns else [hashlib.sha256(str(p).encode('utf-8')).hexdigest() for p in test_frame['image_path']],
        'true_label': [LABELS[int(i)] for i in labels_np],
        'pred_label': [LABELS[int(i)] for i in scores_np.argmax(axis=1)],
        'source': 'real',
        'split': 'test',
        'group_id': test_frame['group_id'].astype(str).to_list(),
        'dataset_source': test_frame['dataset_source'].astype(str).to_list() if 'dataset_source' in test_frame.columns else [''] * len(test_frame),
        'prompt_family_id': [''] * len(test_frame),
    })
    for _index, _class in enumerate(LABELS):
        predictions_table['p_' + _class] = scores_np[:, _index]
    assert predictions_table['image_id'].is_unique
    export_file = out / 'FINAL_real_test_predictions.csv'
    predictions_table.to_csv(export_file, index=False)
    print('Saved paired-evaluation CSV:', export_file, 'rows:', len(predictions_table))


In [ ]:
# Persist reproducible checkpoint/validation history and hashes for later Kaggle evaluation.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import training_run_report
report=training_run_report(model_output=OUT_DIR,manifest_csv=MANIFEST_CSV,
    model_name='PanDerm_Base_LP',
    experiment=EXPERIMENT,seed=SEED,repo_root=DERMSYNREAL_REPO)
print('Artifact inventory:',Path(OUT_DIR)/'artifact_inventory.csv')
print('Run logs:',Path(OUT_DIR)/'run_log.jsonl')
print('Checkpoint provenance:',report['artifacts_sha256'])

### PanDerm provenance
The official encoder is from `SiyuanYan1/PanDerm`, with license restrictions. It is a **frozen-feature linear probe** baseline; it is not full-backbone fine-tuning. See official repo for its separately maintained full fine-tuning code.